In [1]:
!pip install datasets transformers torch

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import json

In [2]:
# Charger le fichier qu'on a sauvegardé avant
with open("donnees_propres.json", "r") as f:
    donnees = json.load(f)

print(f"Nombre d'exemples chargés : {len(donnees)}")
print(f"Exemple 1 — nombre de chunks : {donnees[0]['nb_chunks']}")
print(f"Résumé : {donnees[0]['resume'][:200]}")

FileNotFoundError: [Errno 2] No such file or directory: 'donnees_propres.json'

In [3]:
from datasets import load_dataset
import re
import json

# Recharger BookSum
dataset = load_dataset("kmfoda/booksum")

# Fonctions de nettoyage
def nettoyer_texte(texte):
    texte = texte.strip()
    texte = re.sub(r'\s+', ' ', texte)
    texte = re.sub(r'[^\x00-\x7F]+', '', texte)
    return texte

def decouper_en_chunks(texte, taille=512):
    mots = texte.split()
    chunks = []
    for i in range(0, len(mots), taille):
        chunk = ' '.join(mots[i:i+taille])
        chunks.append(chunk)
    return chunks

def preparer_dataset(dataset, split="train", max_exemples=100):
    donnees = []
    for i in range(min(max_exemples, len(dataset[split]))):
        exemple = dataset[split][i]
        texte = nettoyer_texte(exemple["chapter"])
        resume = nettoyer_texte(exemple["summary_text"])
        chunks = decouper_en_chunks(texte)
        donnees.append({
            "chunks": chunks,
            "resume": resume,
            "nb_chunks": len(chunks)
        })
    return donnees

# Préparer et sauvegarder
donnees = preparer_dataset(dataset)
with open("donnees_propres.json", "w") as f:
    json.dump(donnees, f)

print(f"✅ {len(donnees)} exemples prêts !")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/295M [00:00<?, ?B/s]

dev.csv:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/43.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9600 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1484 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1431 [00:00<?, ? examples/s]

✅ 100 exemples prêts !


In [4]:
!pip install datasets transformers torch

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import json

In [5]:
# Charger le fichier qu'on a sauvegardé avant
with open("donnees_propres.json", "r") as f:
    donnees = json.load(f)

print(f"Nombre d'exemples chargés : {len(donnees)}")
print(f"Exemple 1 — nombre de chunks : {donnees[0]['nb_chunks']}")
print(f"Résumé : {donnees[0]['resume'][:200]}")

Nombre d'exemples chargés : 100
Exemple 1 — nombre de chunks : 14
Résumé : Before any characters appear, the time and geography are made clear. Though it is the last war that England and France waged for a country that neither would retain, the wilderness between the forces 


In [6]:
def chunking_avec_overlap(texte, taille=512, overlap=50):
    """
    taille  = nombre de mots par chunk
    overlap = nombre de mots qui se répètent entre deux chunks
    """
    mots = texte.split()
    chunks = []
    i = 0
    while i < len(mots):
        # Prendre un chunk de 'taille' mots
        chunk = mots[i : i + taille]
        chunks.append(' '.join(chunk))
        # Avancer de (taille - overlap) pour créer le chevauchement
        i += taille - overlap
    return chunks

# Tester sur le premier exemple
texte_test = donnees[0]['chunks'][0]  # premier chunk du premier livre
chunks_overlap = chunking_avec_overlap(texte_test, taille=128, overlap=20)

print(f"Nombre de chunks avec overlap : {len(chunks_overlap)}")
print(f"\nFin du chunk 1 : ...{chunks_overlap[0][-100:]}")
print(f"\nDébut du chunk 2 : {chunks_overlap[1][:100]}...")
print("\n✅ Tu vois que les deux chunks partagent des mots en commun !")

Nombre de chunks avec overlap : 5

Fin du chunk 1 : ...rage in a more martial conflict. But, emulating the patience and self-denial of the practised native

Début du chunk 2 : opportunity to exhibit their courage in a more martial conflict. But, emulating the patience and sel...

✅ Tu vois que les deux chunks partagent des mots en commun !


In [7]:
# On utilise le tokenizer de T5 (c'est ce que ta camarade va utiliser)
tokenizer = AutoTokenizer.from_pretrained("t5-small")

# Tester la tokenisation d'un chunk
chunk_exemple = chunks_overlap[0]
tokens = tokenizer(
    chunk_exemple,
    max_length=512,
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)

print("Forme des input_ids :", tokens["input_ids"].shape)
print("Premier token :", tokens["input_ids"][0][:10])

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Forme des input_ids : torch.Size([1, 512])
Premier token : tensor([   96, 12858,    15,     3,  2741,    19,   539,     6,    11,    82])


In [8]:
class BookSumDataset(Dataset):
    def __init__(self, donnees, tokenizer, max_length=512):
        self.donnees = donnees
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        # Combien d'exemples on a
        return len(self.donnees)

    def __getitem__(self, idx):
        exemple = self.donnees[idx]

        # Prendre le texte complet (tous les chunks joints)
        texte = ' '.join(exemple['chunks'])
        resume = exemple['resume']

        # Tokeniser le texte (entrée)
        entree = self.tokenizer(
            texte,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        # Tokeniser le résumé (sortie attendue)
        sortie = self.tokenizer(
            resume,
            max_length=128,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": entree["input_ids"].squeeze(),
            "attention_mask": entree["attention_mask"].squeeze(),
            "labels": sortie["input_ids"].squeeze()
        }

print("✅ Classe BookSumDataset définie !")

✅ Classe BookSumDataset définie !


In [9]:
# Créer le dataset
train_dataset = BookSumDataset(donnees[:80], tokenizer)
val_dataset   = BookSumDataset(donnees[80:], tokenizer)

# Créer les DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False)

print(f"Taille train dataset : {len(train_dataset)} exemples")
print(f"Taille val dataset   : {len(val_dataset)} exemples")
print(f"Nombre de batches train : {len(train_loader)}")

# Tester un batch
batch = next(iter(train_loader))
print(f"\nForme d'un batch :")
print(f"  input_ids    : {batch['input_ids'].shape}")
print(f"  attention_mask: {batch['attention_mask'].shape}")
print(f"  labels       : {batch['labels'].shape}")

Taille train dataset : 80 exemples
Taille val dataset   : 20 exemples
Nombre de batches train : 20

Forme d'un batch :
  input_ids    : torch.Size([4, 512])
  attention_mask: torch.Size([4, 512])
  labels       : torch.Size([4, 128])


In [10]:
# Sauvegarder le dataset préparé
torch.save(train_dataset, "train_dataset.pt")
torch.save(val_dataset,   "val_dataset.pt")

print("✅ DataLoader prêt !")
print("✅ train_dataset.pt sauvegardé")
print("✅ val_dataset.pt sauvegardé")
print("\nTa camarade peut maintenant utiliser ces datasets pour le fine-tuning !")

✅ DataLoader prêt !
✅ train_dataset.pt sauvegardé
✅ val_dataset.pt sauvegardé

Ta camarade peut maintenant utiliser ces datasets pour le fine-tuning !
